# 🔬 Hallucination Sentinel — Colab + ngrok
### EigenScore-based hallucination detection · `facebook/opt-6.7b`
---
**First-time setup** (run all cells top to bottom):
Upload ZIP → install deps → patch CORS → add ngrok token → launch server

**Subsequent sessions** (skip Step 2A, use Step 2B instead):
Mount Drive → copy project from Drive → install deps → launch server


## Step 1 — Mount Google Drive + Smart Model Cache
- Model weights are always **saved to Drive** (persists across sessions)
- If weights already exist on Drive, they are **copied to local disk** first
- Loading from local disk is ~3× faster than loading from Drive directly

| Scenario | Approx. time |
|----------|--------------|
| First run — fresh download (~13 GB) | 15–30 min |
| Later runs — copy Drive → local, then load | 5–8 min |


In [ ]:
from google.colab import drive
import os, shutil, time

drive.mount('/content/drive')

DRIVE_CACHE = '/content/drive/MyDrive/hf_cache'
LOCAL_CACHE = '/content/hf_cache'
os.makedirs(DRIVE_CACHE, exist_ok=True)

if os.path.exists(DRIVE_CACHE) and os.listdir(DRIVE_CACHE):
    if not os.path.exists(LOCAL_CACHE):
        print('Cached weights found on Drive. Copying to local disk...')
        t0 = time.time()
        shutil.copytree(DRIVE_CACHE, LOCAL_CACHE)
        print(f'Copy done in {(time.time()-t0)/60:.1f} min.')
    else:
        print('Local cache already exists. Skipping copy.')
    ACTIVE_CACHE = LOCAL_CACHE
else:
    print('No cached weights found. Model will download to Drive on first run (~13 GB).')
    ACTIVE_CACHE = DRIVE_CACHE

os.environ['HF_HOME']               = ACTIVE_CACHE
os.environ['TRANSFORMERS_CACHE']    = ACTIVE_CACHE
os.environ['HUGGINGFACE_HUB_CACHE'] = ACTIVE_CACHE
print(f'Active cache: {ACTIVE_CACHE}')


## Step 2A — Upload Project ZIP *(first time only)*
Skip this cell in future sessions and use **Step 2B** instead.


In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()   # select hallucination-detection.zip
zip_name = list(uploaded.keys())[0]

PROJECT_DIR = '/content/hallucination-detection'
os.makedirs(PROJECT_DIR, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('/content/')

os.chdir(PROJECT_DIR)
print('Extracted to:', PROJECT_DIR)
print('Contents:', os.listdir(PROJECT_DIR))

# Save project to Drive so future sessions skip the upload
DRIVE_PROJECT = '/content/drive/MyDrive/hallucination-detection'
if not os.path.exists(DRIVE_PROJECT):
    shutil.copytree(PROJECT_DIR, DRIVE_PROJECT,
                    ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
    print('Project saved to Google Drive. Use Step 2B in future sessions.')


## Step 2B — Load Project from Drive *(subsequent sessions)*
Run this instead of Step 2A once you have done the first-time setup.


In [ ]:
import shutil, os

DRIVE_PROJECT = '/content/drive/MyDrive/hallucination-detection'
PROJECT_DIR   = '/content/hallucination-detection'

if not os.path.exists(DRIVE_PROJECT):
    raise FileNotFoundError('Project not found on Drive. Run Step 2A first.')

if not os.path.exists(PROJECT_DIR):
    shutil.copytree(DRIVE_PROJECT, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('Project loaded from Drive:', PROJECT_DIR)


## Step 3 — Verify GPU & Install Dependencies

In [ ]:
!nvidia-smi


In [ ]:
!pip install -q \
    'torch>=2.2.0' \
    'transformers>=4.40.0' \
    'accelerate>=0.29.0' \
    'bitsandbytes>=0.43.0' \
    'sentence-transformers>=2.7.0' \
    'datasets>=2.19.0' \
    'rouge-score>=0.1.2' \
    'scikit-learn>=1.4.0' \
    'numpy>=1.26.0' \
    'fastapi>=0.111.0' \
    'uvicorn[standard]>=0.29.0' \
    'pydantic>=2.7.0' \
    'pyngrok>=7.0.0'
print('All packages installed.')


## Step 4 — Patch CORS for ngrok
Allows the React frontend to call the API via the ngrok public URL.


In [ ]:
import os, re

server_path = os.path.join(PROJECT_DIR, 'api', 'server.py')
with open(server_path, 'r') as f:
    src = f.read()

patched = re.sub(
    r'allow_origins=\[.*?\],',
    'allow_origins=["*"],  # patched for ngrok',
    src, flags=re.DOTALL
)

if patched != src:
    with open(server_path, 'w') as f:
        f.write(patched)
    print('CORS patched.')
else:
    print('CORS already patched.')


## Step 5 — Configure ngrok Auth Token
Get your free token at https://dashboard.ngrok.com/get-started/your-authtoken


In [ ]:
NGROK_AUTH_TOKEN = 'PASTE_YOUR_NGROK_TOKEN_HERE'  # <-- paste here

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTH_TOKEN
print('ngrok configured.')


## Step 6 — Launch FastAPI Server + ngrok Tunnel
The cell polls `/health` every 5 s and prints the public URL once the model is ready.


In [ ]:
import threading, subprocess, time, os, sys, urllib.request, json

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

PORT     = 8000
LOG_FILE = '/tmp/uvicorn.log'

!pkill -f uvicorn 2>/dev/null; sleep 1
from pyngrok import ngrok as _ngrok
_ngrok.kill()

def _run_server():
    env = os.environ.copy()
    with open(LOG_FILE, 'w') as log:
        subprocess.run(
            [sys.executable, '-m', 'uvicorn', 'api.server:app',
             '--host', '0.0.0.0', '--port', str(PORT), '--log-level', 'info'],
            stdout=log, stderr=log, cwd=PROJECT_DIR, env=env
        )

threading.Thread(target=_run_server, daemon=True).start()
print('Server starting... polling /health every 5s')

ready = False
for i in range(120):
    time.sleep(5)
    elapsed = (i + 1) * 5
    try:
        with urllib.request.urlopen(f'http://localhost:{PORT}/health', timeout=3) as r:
            if json.loads(r.read()).get('model_loaded'):
                print(f'Server ready after {elapsed}s!')
                ready = True
                break
    except Exception:
        if i % 6 == 0:
            print(f'  still loading... {elapsed}s elapsed')

if not ready:
    print('Startup failed. Last log:')
    with open(LOG_FILE) as lf: print(lf.read()[-3000:])
    raise RuntimeError('Server did not start.')

from pyngrok import ngrok
tunnel = ngrok.connect(PORT, 'http')
PUBLIC_URL = tunnel.public_url

print()
print('=' * 60)
print('  PUBLIC URL  :', PUBLIC_URL)
print('  /health     :', PUBLIC_URL + '/health')
print('  POST/analyze:', PUBLIC_URL + '/analyze')
print('=' * 60)
print('Set in your React .env:  VITE_API_URL=' + PUBLIC_URL)


## Step 7 — Smoke Test

In [ ]:
import urllib.request, json

with urllib.request.urlopen(PUBLIC_URL + '/health') as r:
    print('Health:', json.loads(r.read()))


In [ ]:
import urllib.request, json

QUESTION = 'What is the capital of France?'
payload  = json.dumps({'question': QUESTION, 'k': 5, 'use_clipping': True}).encode()
req = urllib.request.Request(
    PUBLIC_URL + '/analyze', data=payload,
    headers={'Content-Type': 'application/json'}, method='POST'
)
with urllib.request.urlopen(req, timeout=300) as r:
    res = json.loads(r.read())

print('Question  :', QUESTION)
print('Verdict   :', res['verdict'])
print('EigenScore:', res['eigenscore'])
print('Threshold :', res['threshold'])
print('Confidence:', res['confidence'])
print('Response  :', res['canonical_response'])


## Step 8 — (Optional) Run Dataset Pipeline
`--limit 50` processes 50 TruthfulQA questions. Remove the flag for all 817.  
Results save incrementally to `data/tfulresults.csv` and support resume.


In [ ]:
!cd "{PROJECT_DIR}" && python pipeline/run_dataset.py --limit 50


## Utilities

In [ ]:
# View server log
!tail -80 /tmp/uvicorn.log


In [ ]:
# List active tunnels
from pyngrok import ngrok
for t in ngrok.get_tunnels():
    print(t.public_url, '->', t.config['addr'])


In [ ]:
# Clean shutdown
from pyngrok import ngrok
ngrok.kill()
!pkill -f uvicorn
print('Stopped.')


In [ ]:
# Delete model weights from Drive (frees ~13 GB)
import shutil
shutil.rmtree('/content/drive/MyDrive/hf_cache', ignore_errors=True)
print('Model weights deleted from Google Drive.')
